# Best practices lab

This notebook turns production guardrail practices into a deterministic simulation for an internal HR/IT support assistant. Every detector, fixture, and release result is synthetic and offline; nothing here calls a live model or service. The examples focus on evidence and failure behavior rather than vendor-specific APIs.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from best_practices_lab import (
    Budget,
    BudgetExceeded,
    ChangeRequest,
    CircuitBreaker,
    ControlMapping,
    Decision,
    LayerResult,
    ReleaseGate,
    ReviewPacket,
    guarded_detector_call,
    gate_passed,
    layered_release_decision,
    load_json,
    load_risk_register,
    misplaced_controls,
    review_change,
    validate_packet,
    validate_risk_register,
)

FIXTURES = Path("fixtures")
print("deterministic simulation:", FIXTURES)


## 1. Risk register

The risk register makes ownership, detection, response, and residual risk explicit before implementation details are chosen. Notice that the threshold-change row exceeds the appetite and has no accepting owner, so validation produces a concrete issue instead of silently treating the risk as complete. A row with explicit acceptance can remain visible while documenting who owns the exception.

In [ ]:
entries = load_risk_register(FIXTURES / "risk_register.json", appetite=2)
risk_issues = validate_risk_register(entries, appetite=2)
[(entry.risk_id, entry.residual, entry.accepted_by) for entry in entries], risk_issues


## 2. Policy, implementation, and boundaries

Policy statements describe the protection, while mappings show where enforcement actually lives. Identity, authorization, and limits cannot be made reliable by prompt wording alone; the lab flags those misplaced mappings. The detector names in this course are deterministic stand-ins for tested production components, not live models.

In [ ]:
mappings = [
    ControlMapping("authenticate the caller", "identity", "prompt"),
    ControlMapping("authorize payroll writes", "authorization", "app"),
    ControlMapping("limit tool calls", "limit", "gateway"),
    ControlMapping("separate retrieved data from instructions", "content", "prompt"),
]
[(mapping.statement, mapping.layer) for mapping in misplaced_controls(mappings)]


## 3. Budgets and circuit breakers

Budgets bound ordinary work such as tool calls, retries, and generated tokens, while a circuit breaker bounds repeated detector failures. This simulation opens after three consecutive failures and fails closed for an irreversible payroll write but allows a read-only request with an audit reason. A successful detector call resets consecutive failures.

In [ ]:
budget = Budget(max_tool_calls=2, max_retries=1, max_tokens=100)
budget.consume("tool_calls")
budget.consume("tokens", 40)
try:
    budget.consume("tool_calls", 2)
except BudgetExceeded as error:
    budget_result = f"blocked: {error.kind}"

breaker = CircuitBreaker(failure_threshold=3)
def unavailable(_: str) -> float:
    raise RuntimeError("synthetic outage")

for _ in range(3):
    irreversible = guarded_detector_call(breaker, unavailable, "payroll write", True)
read_only = guarded_detector_call(breaker, unavailable, "read ticket", False)
budget_result, breaker.is_open, irreversible, read_only


## 4. Protect the guardrail

Changing a threshold, allowlist, or policy changes the safety boundary and therefore needs independent review by the guardrail owner. The author cannot review their own safety-critical change, and a missing or unauthorized reviewer blocks it. Prompt-copy changes still receive an audit outcome but are not treated as safety-critical policy changes.

In [ ]:
changes = [ChangeRequest(**row) for row in load_json(FIXTURES / "change_requests.json")]
[(change.change_id, review_change(change).decision.value, review_change(change).reason_codes) for change in changes]


## 5. Layered evaluation

Unit, component, end-to-end, adversarial, and monitoring layers answer different questions about release safety. The benchmark score of 0.97 is useful evidence, but the poisoned-knowledge-base regression in the adversarial layer still blocks release. This prevents an aggregate benchmark from masking a consequential workflow failure.

In [ ]:
evaluation = load_json(FIXTURES / "evaluation_layers.json")
layers = [LayerResult(**row) for row in evaluation["layers"]]
layer_outcome = layered_release_decision(layers, evaluation["benchmark_score"])
layer_outcome


## 6. Human review packet

Reviewers need the proposed action, evidence identifiers, uncertainty, consequences, and rollback path in one packet. A context-free approval request is rejected because a reviewer cannot assess what will happen or how to undo it. The complete synthetic packet is suitable for a consequential decision review.

In [ ]:
incomplete = ReviewPacket({}, [], "", "", "")
complete = ReviewPacket(
    {"tool": "issue_payroll_adjustment", "preview": "operation-42"},
    ["decision-42", "policy-v2"],
    "detector score is below threshold but the tool is irreversible",
    "payroll state could change for one employee",
    "reverse operation-42 after approval",
)
validate_packet(incomplete), validate_packet(complete)


## 7. Evidence-backed release gate

The release gate checks each checklist item against a manifest and returns the evidence path it inspected. The passing manifest is evaluated with a remediated risk register and passing layers; the failing manifest deliberately uses prompt authorization, lacks a false-negative dataset, and has an untested appeal path. Per-item results make a checklist auditable rather than a collection of unchecked claims.

In [ ]:
gate = ReleaseGate()
manifest_pass = load_json(FIXTURES / "manifest_pass.json")
manifest_fail = load_json(FIXTURES / "manifest_fail.json")
passing_layers = [
    LayerResult(name, True, "synthetic pass")
    for name in ("unit", "component", "end_to_end", "adversarial", "monitoring")
]
passing_layer_outcome = layered_release_decision(passing_layers, 0.97)
pass_items = gate.evaluate(manifest_pass, [], passing_layer_outcome)
fail_items = gate.evaluate(manifest_fail, [], passing_layer_outcome)
{
    "pass": (gate_passed(pass_items), [(item.item, item.evidence) for item in pass_items]),
    "failures": [item.item for item in fail_items if not item.passed],
}


## 8. Summary

The lab connects risk ownership to control placement, bounded failure behavior, independent change review, layered evaluation, human evidence, and release evidence. The narrowest useful improvement is the one that turns a failed assertion into a concrete control or test, rather than raising a benchmark threshold without understanding the failure. Use the exercises below to extend the same deterministic simulation.

In [ ]:
summary = {
    "risk_issues": len(risk_issues),
    "misplaced_controls": len(misplaced_controls(mappings)),
    "benchmark": evaluation["benchmark_score"],
    "manifest_passed": gate_passed(pass_items),
}
summary


## Exercises

1. Add a risk-register row for a new HR/IT asset and decide whether its residual risk needs explicit acceptance.
2. Add an authorized reviewer for a safety-critical change and compare the result with a prompt-copy change.
3. Repair the adversarial regression and the three failing manifest items, then explain why the benchmark score alone was insufficient.